# 04 – Material Classification Training

This notebook trains an **EfficientNetV2-S** classifier on the material crops
produced in notebook 03.  The model maps each crop to one of five classes:
`plastic`, `glass`, `metal`, `paper`, `other`.

**Output:** `models/classification/best_classifier.pt`

## 1 · Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torch import nn

CROPS_DIR   = REPO_ROOT / "datasets" / "processed" / "classification" / "crops"
MODELS_DIR  = REPO_ROOT / "models" / "classification"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Crops  : {CROPS_DIR}")

## 2 · Dataset Preparation

The crops directory must follow the `ImageFolder` convention:

```
crops/
├── plastic/
├── glass/
├── metal/
├── paper/
└── other/
```

We do an 80/20 train/val split at the **file** level (crops from the same
video are already isolated by the group-aware split in notebook 01, so no
further group constraint is needed here).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

CLASS_NAMES = ["glass", "metal", "other", "paper", "plastic"]  # sorted alphabetically
NUM_CLASSES = len(CLASS_NAMES)

# ── augmentation transforms defined in Section 3 ─────────────────────────────
# (referenced here; defined below)
BATCH_SIZE = 32
VAL_SPLIT  = 0.20

print(f"Classes     : {CLASS_NAMES}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Val fraction: {VAL_SPLIT}")

## 3 · Augmentations

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Load full dataset with training transforms; swap val subset to val_transform
if CROPS_DIR.exists():
    full_dataset = datasets.ImageFolder(str(CROPS_DIR), transform=train_transform)
    print(f"Total samples : {len(full_dataset)}")
    print(f"Classes found : {full_dataset.classes}")

    n_val   = int(len(full_dataset) * VAL_SPLIT)
    n_train = len(full_dataset) - n_val
    train_ds, val_ds = random_split(
        full_dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(42),
    )

    # Apply val transform to the validation subset
    val_ds.dataset = datasets.ImageFolder(str(CROPS_DIR), transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    print(f"Train samples : {n_train}  |  Val samples: {n_val}")
else:
    print(f"Crops directory not found: {CROPS_DIR}")
    print("Complete notebooks 02 and 03 first.")
    train_loader = val_loader = None

## 4 · Model

We use **EfficientNetV2-S** pretrained on ImageNet and replace the
classifier head with a linear layer matching our number of classes.

In [ ]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

# Replace the classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)

model = model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")

## 5 · Training Loop

Class weights are computed from the dataset to handle class imbalance.

In [ ]:
from sklearn.metrics import f1_score

EPOCHS    = 30
LR        = 1e-4
CKPT_PATH = MODELS_DIR / "best_classifier.pt"

def compute_class_weights(dataset: datasets.ImageFolder) -> torch.Tensor:
    counts = np.bincount([s[1] for s in dataset.samples])
    weights = 1.0 / (counts + 1e-6)
    return torch.tensor(weights / weights.sum(), dtype=torch.float32)

if train_loader is not None:
    class_weights = compute_class_weights(full_dataset).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_f1 = 0.0
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, EPOCHS + 1):
        # ── train ────────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= n_train

        # ── validate ─────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, labels).item() * imgs.size(0)
                preds = logits.argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        val_loss /= n_val
        val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        val_f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), CKPT_PATH)

        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.3f} | "
            f"val_F1={val_f1:.3f}" +
            (" ← best" if val_f1 == best_val_f1 else "")
        )

    print(f"\nBest val macro-F1: {best_val_f1:.4f}")
    print(f"Checkpoint saved → {CKPT_PATH}")
else:
    print("Skipped – no data loaded.")

## 6 · Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

if train_loader is not None and CKPT_PATH.exists():
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Validation Set)")
    plt.tight_layout()
    plt.show()
else:
    print("Model checkpoint not found – run Section 5 first.")

## 7 · Save Model Checkpoint

In [ ]:
if train_loader is not None:
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "num_classes": NUM_CLASSES,
        "best_val_f1": best_val_f1,
    }
    full_ckpt_path = MODELS_DIR / "classifier_checkpoint_full.pt"
    torch.save(checkpoint, full_ckpt_path)
    print(f"Full checkpoint saved → {full_ckpt_path}")
else:
    print("Skipped – no trained model.")